In [38]:
# 필요한 패키지 설치
# !pip install folium

# folium은 Python에서 Leaflet.js 기반의 인터랙티브 지도(웹 기반)를 만들 수 있게 해주는 라이브러리. 
# 주로 **위치 기반 데이터 시각화(GIS)**를 할 때 많이 사용되며, 복잡한 자바스크립트 없이도 지도를 웹에 표시하고, 마커나 원, 경로, 히트맵 등을 쉽게 추가할 수 있.

import folium
from folium import Marker
from folium import plugins
from folium import GeoJson
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.graph_objects as go

plt.rcParams['figure.dpi'] = 300

In [49]:
# 서울 스타벅스 지점 데이터 불러오기
# https://www.kaggle.com/datasets/sewonghwang/starbucks-seoul
df=pd.read_csv("../main/datasets/Starbucks_Seoul.csv")

# 지역 구분을 위한 json 파일 불러오기
geo="../main/datasets/Seoul_Gu.json" 

# 데이터 샘플 확인
df.head()

# latitude 위도	longitude 경도

,name,address,gu_name,latitude,longitude
0,GS타워,서울특별시 강남구 논현로 508 (역삼동),강남구,37.501859,127.037278
1,SSG마켓도곡R,"서울특별시 강남구 언주로30길 57, 타워팰리스Ⅱ F 지하1층 (도곡동)",강남구,37.490298,127.054895
2,W-Mall,서울특별시 금천구 디지털로 188 (가산동),금천구,37.477305,126.887691
3,가든파이브,서울특별시 송파구 충민로 10 (문정동) 가든파이브툴,송파구,37.478232,127.119370
4,가락본동,서울특별시 송파구 송파대로30길 13 (가락동),송파구,37.494895,127.118785


In [40]:
df.shape

(521, 5)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       521 non-null    object 
 1   address    521 non-null    object 
 2   gu_name    521 non-null    object 
 3   latitude   521 non-null    float64
 4   longitude  521 non-null    float64
dtypes: float64(2), object(3)
memory usage: 20.5+ KB


In [42]:
df.isnull().sum()

name         0
address      0
gu_name      0
latitude     0
longitude    0
dtype: int64

In [43]:
# 기본 지도 시각화 (서울의 위도, 경도 입력)

m = folium.Map(location=[37.541, 126.986], zoom_start=12)
m

In [44]:
# 지도 형태 변경
m = folium.Map(
    location=[37.541, 126.986],
    tiles='OpenStreetMap',
    attr='Map tiles by Stamen Design, under CC BY 3.0. Data by OpenStreetMap, under ODbL.',
    zoom_start=12
)

# tiles

# folium에서 사용할 수 있는 다양한 타일 스타일은 지도 시각화의 목적이나 분위기에 따라 선택할 수 있으며, 대부분은 외부 타일 서버를 활용. 

# ✅ 주요 타일 스타일 목록

# 타일 이름	                tiles 값	                                특징
# OpenStreetMap	        'OpenStreetMap'	                        기본값, 가장 일반적인 지도 스타일
# Stamen Terrain	        'Stamen Terrain'	                    지형, 산맥, 강 등 강조 (등산, 지리용)
# Stamen Toner	        'Stamen Toner'	                        흑백, 대비 강함 (분석용 지도 배경)
# Stamen Watercolor	    'Stamen Watercolor'	                    수채화 느낌, 예술적인 배경
# CartoDB Positron	    'CartoDB positron'	                    밝고 미니멀한 스타일 (데이터 시각화에 적합)
# CartoDB Dark_Matter	    'CartoDB dark_matter'	                어두운 배경, 야간 모드에 적합
# NASAGIBS Blue Marble	사용자 지정 (링크 필요)	                     NASA 위성 이미지 기반 타일 (심미용)



# 원하는 좌표에 반경(radius) 표시 (남산)
folium.CircleMarker([37.5538, 126.9810],radius=50, 
                    popup='Laurelhurst Park', color='#3246cc', 
                    fill_color='#3246cc').add_to(m)

# 원하는 좌표에 포인트 표시 (남산)
folium.Marker([37.5538, 126.9810], popup='The Waterfront').add_to(m)
    
m

In [45]:
# 서울 지도에 스타벅스 지점 수 시각화

# 지도 객체 생성 (width와 height를 올바르게 지정)
m = folium.Map(
    location=[37.541, 126.986],
    zoom_start=12,
    width="100%",    # ✅ 문자열로 백분율 표현
    height="100%"    # ✅ 올바른 형식으로 수정
)

# 위치 리스트 생성
locations = list(zip(df.latitude, df.longitude))

# 마커 클러스터 생성
cluster = plugins.MarkerCluster(
    locations=locations,
    popups=df["name"].tolist()
)

# 클러스터 지도에 추가
m.add_child(cluster)

# 지도 출력
m
m

In [46]:
# 서울 지도에 스타벅스 지점수 도드맵 시각화

m = folium.Map(
    location=[37.541, 126.986], 
    zoom_start=12, 
    width="100%", 
    height="100%"
    )

locations = list(
    zip(
        df.latitude, 
        df.longitude
        )
)

for i in range(len(locations)):
    folium.CircleMarker(
        location=locations[i],
        radius=1
        ).add_to(m)
m

In [47]:
# 서울 구 별 스타벅스 지점 수 집계 및 중심점 산출
df_m = df.groupby('gu_name').agg({'latitude':'mean',
                                  'longitude':'mean',
                                  'name':'count'}).reset_index()
df_m.head()

,gu_name,latitude,longitude,name
0,강남구,37.507603,127.044611,80
1,강동구,37.539914,127.137106,14
2,강북구,37.626866,127.026372,5
3,강서구,37.555716,126.841528,16
4,관악구,37.481759,126.944286,11


In [53]:
# 서울 구 별 스타벅스 지점 수 버블맵 시각화

# 기본 지도 생성
m = folium.Map(location=[37.541, 126.986], tiles='Cartodb Positron', 
               zoom_start=11, width="100%", 
               height="100%")

# GeoJSON 파일을 적절한 인코딩으로 불러오기
import json
with open("../main/datasets/Seoul_Gu.json", encoding="cp949") as f:
    geo = json.load(f)

# 구별 구분선, 색상 설정
folium.Choropleth(
    geo_data=geo, # 앞에서 불러온 json 파일 적용
    fill_color="gray"
    ).add_to(m)

# 버블맵 삽입
locations = list(zip(df_m.latitude, df_m.longitude))
for i in range(len(locations)):
    row = df_m.iloc[i]
    folium.CircleMarker(location=locations[i],
                        radius= float(row.name/2), # 버블 크기 설정
                        fill_color="blue"
                       ).add_to(m)
m

# radius에 카운팅을 넣어줌
# 만약 원이 커서 겹치면 float(row.name/1)의 분모값을 조정

In [58]:
# 미국 실업률 정보의 코로플레스맵 시각화를 위한 데이터, json 불러오기
# https://www.kaggle.com/datasets/sewonghwang/us-unemployment
df2 =pd.read_csv("../main/datasets/us_states_unemployment.csv")

# 주 별 경계 json 파일 불러오기
us_geo = '../main/datasets/folium_us-states.json'

df2.head()

,State,Unemployment
0,AL,7.1
1,AK,6.8
2,AZ,8.1
3,AR,7.2
4,CA,10.1


In [63]:
# 미국 주별 실업률 코로플레스맵 시각화

# 미국 지도 시각화
m = folium.Map(
    location=[40, -98],  # 미국 중심 좌표
    zoom_start=4,        # 줌 레벨 (낮을수록 넓은 지역 표시)
    tiles="CartoDB positron"  # 밝고 미니멀한 배경지도
)

# 지도에 주 경계선, 실업률 데이터 연동
folium.Choropleth(
    geo_data=us_geo,  # 미국 주 경계가 포함된 GeoJSON 객체 또는 파일 경로
    data=df2,         # 실업률 데이터가 들어있는 DataFrame
    columns=['State', 'Unemployment'],  # [주 코드, 실업률] 컬럼
    key_on='feature.id',  # GeoJSON의 feature.id와 df2의 'State'를 연결
    fill_color='YlGn',    # 색상 팔레트: 연두~녹색 계열
    fill_opacity=0.7,     # 면 색상 투명도
    line_opacity=0.2,     # 경계선 투명도
    legend_name='실업률 (%)'  # 범례 제목
).add_to(m)

m

In [61]:
# 서울과 각국의 수도 간의 커넥션맵 시각화

# 서울과 도쿄, 워싱턴, 마닐라, 파리, 모스크바 위경도 입력
source_to_dest = zip([37.541,37.541,37.541,37.541,37.541], 
                     [35.6804, 38.9072, 14.5995, 48.8566,55.7558],
                     [126.986,126.986,126.986,126.986,126.986], 
                     [139.7690, -77.0369, 120.9842, 2.3522,37.6173])

fig = go.Figure()

## for 문을 활용하여 위경도 입력
for a, b, c, d in source_to_dest:
    fig.add_trace(go.Scattergeo(
                        lat = [a, b],
                        lon = [c, d],
                        mode = 'lines',
                        line = dict(width = 1, color="red"),
                        opacity = 0.5 # 선 투명도
                        ))

fig.update_layout(
                margin={"t":0,"b":0,"l":0, "r":0, "pad":0},
                showlegend=False,
                geo = dict(
                showcountries=True) # 국가 경계선
                )

fig.show()